---
## Bagian 13 — Kerjakan untuk proyek tim: M2

**M2 — Data Quality Report, bobot 10%.** Tiga berkas yang dikumpulkan:

| Berkas | Isi | Diperiksa oleh rubrik |
|---|---|---|
| `reports/data_quality_report.docx` | Temuan, keputusan, keterbatasan | Kelengkapan diagnosis (25), Justifikasi (30) |
| `notebooks/02_cleaning.ipynb` | Pipeline yang dapat dijalankan ulang | Ketepatan implementasi (20) |
| `cleaning_log.md` | Setiap langkah, jumlah, dan alasan | Dokumentasi dan keterlacakan (15) |

Dimensi kelima, **Penanganan data pribadi (10)**, dinilai dari apakah data olahan Anda tetap aman —
yaitu lanjutan pekerjaan pertemuan 3.

Gunakan `TEMPLATE_Data_Quality_Report.docx` sebagai kerangka laporan.

In [30]:
import pandas as pd
import os
from pathlib import Path


pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 150)

In [31]:
def bersihkan(df_mentah):
    """Terima data mentah, kembalikan data bersih beserta log keputusan."""
    x = df_mentah.copy()
    log = []

    # Fungsi pembantu untuk mencatat log
    def c(langkah, n, alasan):
        log.append({
            "langkah": langkah,
            "baris_terdampak": int(n),
            "alasan": alasan
        })

    # --- LANGKAH 1: Perbaiki tipe data Date dan Time ---
    date_awal = x["Date"].copy()
    time_awal = x["Time"].copy()

    date_baru = pd.to_datetime(
        x["Date"], format="%m/%d/%Y", errors="coerce"
    )
    time_baru = pd.to_datetime(
        x["Time"], format="%I:%M:%S %p", errors="coerce"
    ).dt.time

    gagal_date = int(
        (date_awal.notna() & date_baru.isna()).sum()
    )
    gagal_time = int(
        (time_awal.notna() & pd.Series(time_baru, index=x.index).isna()).sum()
    )

    x["Date"] = date_baru
    x["Time"] = time_baru

    c(
        "Perbaikan tipe data Date dan Time",
        int((date_awal.notna() & date_baru.notna()).sum()),
        f"Konversi tanggal dan waktu; gagal konversi Date={gagal_date}, Time={gagal_time}."
    )

    # --- LANGKAH 2: Hapus spasi pada kolom teks ---
    kolom_teks = x.select_dtypes(
        include=["object", "string"]
    ).columns

    baris_spasi = pd.Series(False, index=x.index)

    for kolom in kolom_teks:
        # Pastikan nilai diperlakukan sebagai teks
        nilai_awal = x[kolom].copy()
        nilai_teks = nilai_awal.astype("string")
        nilai_baru = nilai_teks.str.strip()

        mask_spasi = (
            nilai_awal.notna() &
            nilai_teks.ne(nilai_baru).fillna(False)
        )

        baris_spasi |= mask_spasi
        x[kolom] = nilai_baru

    c(
        "Pembersihan spasi pada kolom teks",
        int(baris_spasi.sum()),
        "Menghapus spasi di awal atau akhir nilai pada kolom teks."
    )

    # --- LANGKAH 3: Periksa placeholder ---
    placeholder = [
        "", "none", "null", "n/a", "na", "-", "tidak ada"
    ]

    baris_placeholder = pd.Series(False, index=x.index)

    for kolom in x.select_dtypes(
        include=["object", "string"]
    ).columns:
        nilai = x[kolom].astype("string").str.strip().str.lower()
        mask = nilai.isin(placeholder)

        baris_placeholder |= mask
        x.loc[mask, kolom] = pd.NA

    c(
        "Pemeriksaan placeholder",
        int(baris_placeholder.sum()),
        "Memeriksa teks yang berpotensi mewakili nilai kosong."
    )

    # --- LANGKAH 4: Normalisasi kategori ---
    # Tidak mengubah label kategori secara otomatis karena
    # perubahan harus berdasarkan temuan dan bukti.
    c(
        "Pemeriksaan konsistensi kategori",
        0,
        "Tidak ada kategori yang diubah tanpa bukti adanya variasi penulisan yang perlu diseragamkan."
    )

    # --- LANGKAH 5: Hapus duplikat persis ---
    n_duplikat = int(x.duplicated().sum())
    x = x.drop_duplicates()

    c(
        "Pemeriksaan duplikat persis",
        n_duplikat,
        "Menghapus baris yang identik pada seluruh kolom jika ditemukan."
    )

    # --- LANGKAH 6: Periksa nilai yang tidak valid ---
    aturan = {
        "Unit price": lambda s: s <= 0,
        "Quantity": lambda s: s < 1,
        "Sales": lambda s: s < 0,
        "cogs": lambda s: s < 0,
        "gross income": lambda s: s < 0,
        "Tax 5%": lambda s: s < 0,
        "Rating": lambda s: (s < 0) | (s > 10),
    }

    jumlah_tidak_valid = 0

    for kolom, aturan_salah in aturan.items():
        if kolom in x.columns:
            nilai = pd.to_numeric(x[kolom], errors="coerce")
            jumlah_tidak_valid += int(aturan_salah(nilai).sum())

    c(
        "Pemeriksaan nilai tidak valid",
        jumlah_tidak_valid,
        "Memeriksa nilai numerik yang melanggar aturan validasi; nilai tidak diubah otomatis."
    )

    # --- LANGKAH 7: Periksa nilai kosong ---
    jumlah_kosong = int(x.isna().sum().sum())

    c(
        "Pemeriksaan nilai kosong",
        jumlah_kosong,
        "Memeriksa nilai kosong setelah konversi tipe data dan placeholder."
    )

    # --- LANGKAH 8: Tambahkan kolom turunan ---
    if "Date" in x.columns:
        x["year_month"] = x["Date"].dt.to_period("M").astype("string")

    c(
        "Penambahan kolom year_month",
        len(x) if "Date" in x.columns else 0,
        "Membuat kolom tahun-bulan untuk membantu analisis tren penjualan."
    )

    # --- LANGKAH 9: Pseudonimisasi Invoice ID ---
    # Hash bukan anonimisasi penuh; dokumentasikan sebagai
    # keputusan perlindungan pengenal transaksi.
    if "Invoice ID" in x.columns:
        invoice_lama = x["Invoice ID"].astype("string")
        invoice_baru = invoice_lama.map(
            lambda v: (
                __import__("hashlib").sha256(
                    ("supermarket_m2:" + v).encode("utf-8")
                ).hexdigest()[:16]
                if pd.notna(v) else pd.NA
            )
        )

        x["Invoice ID"] = invoice_baru

        c(
            "Pseudonimisasi Invoice ID",
            int(invoice_lama.notna().sum()),
            "Mengganti pengenal transaksi dengan hash agar nilai asli tidak langsung ditampilkan."
        )

    df_log = pd.DataFrame(log)

    return x.reset_index(drop=True), df_log

In [35]:
# 13.1 — salin pola workbook ini ke dataset tim
df_tim = pd.read_csv("SuperMarket Analysis_cleaned.csv")

# Jalankan pipeline cleaning M2
bersih_tim, log_tim = bersihkan(df_tim)

# Simpan hasil baru, tanpa menimpa hasil Pertemuan 3
output_path = Path("data/processed/supermarket_clean_m2.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

bersih_tim.to_csv(output_path, index=False)

print("Data awal:", df_tim.shape)
print("Data hasil M2:", bersih_tim.shape)
print("Data tersimpan:", output_path)

display(log_tim)


Data awal: (1000, 17)
Data hasil M2: (1000, 18)
Data tersimpan: data/processed/supermarket_clean_m2.csv


,langkah,baris_terdampak,alasan
0,Perbaikan tipe data Date dan Time,1000,Konversi tanggal dan waktu; gagal konversi Dat...
1,Pembersihan spasi pada kolom teks,0,Menghapus spasi di awal atau akhir nilai pada ...
2,Pemeriksaan placeholder,0,Memeriksa teks yang berpotensi mewakili nilai ...
3,Pemeriksaan konsistensi kategori,0,Tidak ada kategori yang diubah tanpa bukti ada...
4,Pemeriksaan duplikat persis,0,Menghapus baris yang identik pada seluruh kolo...
5,Pemeriksaan nilai tidak valid,0,Memeriksa nilai numerik yang melanggar aturan ...
6,Pemeriksaan nilai kosong,0,Memeriksa nilai kosong setelah konversi tipe d...
7,Penambahan kolom year_month,1000,Membuat kolom tahun-bulan untuk membantu anali...
8,Pseudonimisasi Invoice ID,1000,Mengganti pengenal transaksi dengan hash agar ...


In [ ]:
# 13.2 — jalankan pemeriksa kelengkapan sebelum mengumpulkan
# Dari terminal, di dalam folder repositori tim:
#     python cek_m2.py
# Skrip akan memeriksa keberadaan berkas, struktur folder, dan isi minimal cleaning log.


### Daftar periksa M2

- [ ] Seluruh temuan pertemuan 4 muncul di laporan, termasuk yang Anda putuskan untuk dibiarkan
- [ ] Setiap keputusan punya alasan tertulis, bukan sekadar deskripsi tindakan
- [ ] Minimal satu keputusan menyebutkan **alternatif yang dipertimbangkan dan ditolak**
- [ ] Dampak terhadap sebaran data ditunjukkan untuk minimal satu kolom (pola Bagian 7)
- [ ] Keterbatasan dinyatakan terbuka, termasuk kolom yang tidak bisa diselamatkan
- [ ] `data/raw/` tidak berubah sama sekali dan tidak ikut diunggah
- [ ] Notebook berjalan dari awal sampai akhir tanpa galat setelah *Restart and run all*
- [ ] `cleaning_log.md` ada di akar repositori
- [ ] `python cek_m2.py` berjalan tanpa keluhan

> **Kesalahan yang paling mahal pada M2:** menulis apa yang dilakukan, bukan mengapa.
> "Kolom X diisi dengan median" adalah deskripsi. "Kolom X diisi dengan median karena kekosongannya
> tersebar merata di semua kelompok dan hanya 3%, sehingga pilihan ini tidak menggeser sebaran lebih dari
> 1%; alternatif membuang baris ditolak karena akan menghapus 48 transaksi yang kolom lainnya lengkap"
> adalah justifikasi. Rubrik memberi bobot 30 untuk yang kedua.